# Session 10: Network Security Fundamentals: ACLs and DHCP Snooping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-10-network-security-fundamentals.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-10_acl_engine.py`

In [1]:
"""Session 10 - Evaluate Cisco-style ACLs exactly as a router does.

- Lines are checked top-down; the FIRST match decides (permit or deny).
- Every ACL ends with an invisible "deny any" (implicit deny).
- Wildcard masks: 0 bit = must match, 1 bit = ignore. "host x" = x 0.0.0.0, "any" = 0.0.0.0 255.255.255.255.
- Standard ACLs (1-99) check only the source; extended ACLs (100-199) check
  protocol, source, destination and port.
Run:  python session-10_acl_engine.py
"""
import ipaddress


def ip(x): return int(ipaddress.ip_address(x))


def match_addr(addr, base, wildcard):
    return (ip(addr) & ~ip(wildcard) & 0xFFFFFFFF) == (ip(base) & ~ip(wildcard) & 0xFFFFFFFF)


def parse_addr(tokens):
    """Consume 'any' | 'host A' | 'A W' and return ((base, wildcard), remaining tokens)."""
    if tokens[0] == "any":
        return ("0.0.0.0", "255.255.255.255"), tokens[1:]
    if tokens[0] == "host":
        return (tokens[1], "0.0.0.0"), tokens[2:]
    return (tokens[0], tokens[1]), tokens[2:]


def parse_line(line):
    t = line.split()
    action, rest = t[0], t[1:]
    proto = None
    if rest[0] in ("ip", "tcp", "udp", "icmp"):
        proto, rest = rest[0], rest[1:]
    src, rest = parse_addr(rest)
    dst, port = None, None
    if proto:
        dst, rest = parse_addr(rest)
        if rest[:1] == ["eq"]:
            port = int(rest[1])
    return {"text": line, "action": action, "proto": proto, "src": src, "dst": dst, "port": port}


def evaluate(acl, pkt):
    for n, rule in enumerate(acl, 1):
        if rule["proto"] not in (None, "ip") and rule["proto"] != pkt["proto"]:
            continue
        if not match_addr(pkt["src"], *rule["src"]):
            continue
        if rule["dst"] and not match_addr(pkt["dst"], *rule["dst"]):
            continue
        if rule["port"] is not None and rule["port"] != pkt.get("port"):
            continue
        return rule["action"].upper(), f"line {n}: {rule['text']}"
    return "DENY", "implicit deny any (end of list)"


STANDARD_10 = ["permit 192.168.99.0 0.0.0.255", "deny any"]            # VTY: only the management VLAN
EXTENDED_110 = [
    "permit tcp 192.168.20.0 0.0.0.255 host 192.168.30.10 eq 443",     # students -> LMS over HTTPS
    "permit udp 192.168.20.0 0.0.0.255 host 192.168.30.53 eq 53",      # students -> DNS
    "deny ip 192.168.20.0 0.0.0.255 192.168.10.0 0.0.0.255",            # students -/-> lecturers VLAN
    "permit icmp 192.168.20.0 0.0.0.255 any",                           # ping for troubleshooting
    "permit tcp 192.168.20.0 0.0.0.255 any eq 443",                     # web browsing
]

if __name__ == "__main__":
    std = [parse_line(x) for x in STANDARD_10]
    print("Standard ACL 10 applied to 'line vty 0 4' (who may SSH to the switch):")
    for src in ["192.168.99.25", "192.168.20.14"]:
        print(f"  SSH from {src:15} -> {evaluate(std, {'src': src})}")

    ext = [parse_line(x) for x in EXTENDED_110]
    print("\nExtended ACL 110 inbound on the STUDENTS VLAN interface:")
    tests = [
        {"proto": "tcp", "src": "192.168.20.14", "dst": "192.168.30.10", "port": 443},
        {"proto": "tcp", "src": "192.168.20.14", "dst": "192.168.30.10", "port": 22},
        {"proto": "udp", "src": "192.168.20.14", "dst": "192.168.30.53", "port": 53},
        {"proto": "tcp", "src": "192.168.20.14", "dst": "192.168.10.50", "port": 445},
        {"proto": "icmp", "src": "192.168.20.14", "dst": "8.8.8.8"},
        {"proto": "tcp", "src": "192.168.20.14", "dst": "142.250.1.1", "port": 443},
        {"proto": "tcp", "src": "192.168.20.14", "dst": "142.250.1.1", "port": 80},
    ]
    for p in tests:
        verdict, why = evaluate(ext, p)
        desc = f"{p['proto']} {p['src']} -> {p['dst']}" + (f":{p['port']}" if 'port' in p else "")
        print(f"  {desc:42} {verdict:6} ({why})")
    print("\nWildcard check: 10.1.0.0 0.0.255.255 matches 10.1.200.7?",
          match_addr("10.1.200.7", "10.1.0.0", "0.0.255.255"))

Standard ACL 10 applied to 'line vty 0 4' (who may SSH to the switch):
  SSH from 192.168.99.25   -> ('PERMIT', 'line 1: permit 192.168.99.0 0.0.0.255')
  SSH from 192.168.20.14   -> ('DENY', 'line 2: deny any')

Extended ACL 110 inbound on the STUDENTS VLAN interface:
  tcp 192.168.20.14 -> 192.168.30.10:443     PERMIT (line 1: permit tcp 192.168.20.0 0.0.0.255 host 192.168.30.10 eq 443)
  tcp 192.168.20.14 -> 192.168.30.10:22      DENY   (implicit deny any (end of list))
  udp 192.168.20.14 -> 192.168.30.53:53      PERMIT (line 2: permit udp 192.168.20.0 0.0.0.255 host 192.168.30.53 eq 53)
  tcp 192.168.20.14 -> 192.168.10.50:445     DENY   (line 3: deny ip 192.168.20.0 0.0.0.255 192.168.10.0 0.0.0.255)
  icmp 192.168.20.14 -> 8.8.8.8              PERMIT (line 4: permit icmp 192.168.20.0 0.0.0.255 any)
  tcp 192.168.20.14 -> 142.250.1.1:443       PERMIT (line 5: permit tcp 192.168.20.0 0.0.0.255 any eq 443)
  tcp 192.168.20.14 -> 142.250.1.1:80        DENY   (implicit deny any (end o

## `session-10_dhcp_snooping.py`

In [2]:
"""Session 10 - DHCP snooping: stop rogue DHCP servers and build the binding table.

Ports are UNTRUSTED by default. Server messages (OFFER, ACK) are allowed only
from TRUSTED ports (the uplink towards the real DHCP server). Every ACK that
passes creates a binding (MAC, IP, VLAN, port) that Dynamic ARP Inspection
and IP Source Guard later use to drop spoofed traffic. A rate limit on
untrusted ports slows DHCP starvation floods.
Run:  python session-10_dhcp_snooping.py
"""

TRUSTED = {"Gi0/1"}                 # uplink to the legitimate server
RATE_LIMIT = 10                     # packets per second allowed on untrusted ports


def snoop(events):
    bindings, log, counters = [], [], {}
    for t, port, msg, mac, ip in events:
        counters[(t, port)] = counters.get((t, port), 0) + 1
        if port not in TRUSTED and counters[(t, port)] > RATE_LIMIT:
            log.append(f"t={t} {port}: rate limit exceeded -> port err-disabled")
            continue
        if msg in ("OFFER", "ACK") and port not in TRUSTED:
            log.append(f"t={t} {port}: DROPPED {msg} from {mac} (server message on untrusted port)")
            continue
        log.append(f"t={t} {port}: forwarded {msg} {mac} {ip or ''}".rstrip())
        if msg == "ACK":
            bindings.append((mac, ip, 10, "Fa0/3"))      # client port learned from the DISCOVER
    return bindings, log


def dai_check(bindings, port, mac, ip):
    ok = any(b[0] == mac and b[1] == ip and b[3] == port for b in bindings)
    return "permit" if ok else "DROP (no matching binding)"


if __name__ == "__main__":
    events = [
        (0, "Fa0/3", "DISCOVER", "aaaa.0000.0003", None),
        (0, "Fa0/7", "OFFER", "dead.beef.0007", "10.99.99.50"),     # rogue server on a user port
        (0, "Gi0/1", "OFFER", "0011.2233.4455", "192.168.10.21"),
        (0, "Fa0/3", "REQUEST", "aaaa.0000.0003", "192.168.10.21"),
        (0, "Gi0/1", "ACK", "aaaa.0000.0003", "192.168.10.21"),
    ] + [(1, "Fa0/9", "DISCOVER", f"0bad.0000.{i:04x}", None) for i in range(15)]   # starvation flood
    bindings, log = snoop(events)
    seen = set()
    for line in log:
        if "0bad" in line and "forwarded" in line:
            if "flood" in seen: continue
            seen.add("flood"); line += "   (... more DISCOVERs with random MACs)"
        if "rate limit" in line and "rl" in seen: continue
        if "rate limit" in line: seen.add("rl")
        print(line)
    print("\nshow ip dhcp snooping binding")
    print("MacAddress       IpAddress        Lease  Type           VLAN  Interface")
    for mac, ip_, vlan, port in bindings:
        print(f"{mac:16} {ip_:16} 86400  dhcp-snooping  {vlan:<5} {port}")
    print("\nDynamic ARP Inspection uses the table:")
    print("  ARP from Fa0/3 aaaa.0000.0003 claims 192.168.10.21 ->", dai_check(bindings, "Fa0/3", "aaaa.0000.0003", "192.168.10.21"))
    print("  ARP from Fa0/7 dead.beef.0007 claims 192.168.10.1  ->", dai_check(bindings, "Fa0/7", "dead.beef.0007", "192.168.10.1"))

t=0 Fa0/3: forwarded DISCOVER aaaa.0000.0003
t=0 Fa0/7: DROPPED OFFER from dead.beef.0007 (server message on untrusted port)
t=0 Gi0/1: forwarded OFFER 0011.2233.4455 192.168.10.21
t=0 Fa0/3: forwarded REQUEST aaaa.0000.0003 192.168.10.21
t=0 Gi0/1: forwarded ACK aaaa.0000.0003 192.168.10.21
t=1 Fa0/9: forwarded DISCOVER 0bad.0000.0000   (... more DISCOVERs with random MACs)
t=1 Fa0/9: rate limit exceeded -> port err-disabled

show ip dhcp snooping binding
MacAddress       IpAddress        Lease  Type           VLAN  Interface
aaaa.0000.0003   192.168.10.21    86400  dhcp-snooping  10    Fa0/3

Dynamic ARP Inspection uses the table:
  ARP from Fa0/3 aaaa.0000.0003 claims 192.168.10.21 -> permit
  ARP from Fa0/7 dead.beef.0007 claims 192.168.10.1  -> DROP (no matching binding)
